# Visualisation 

# 1. Imports

In [1]:
import numpy as np
import pandas as pd
import tifffile as tiff
from astropy.io import fits
import h5py

from utils.naming import *
from pathlib import Path
from matplotlib import colormaps

import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ImageNormalize, PercentileInterval, AsinhStretch
import imageio.v2 as imageio

# --- Data locations ----------------------------------------------------------
GHOST = Path("/home/aosimul/noah/data/ghost_images")
RESULTS = Path("/home/aosimul/noah/src/pipeline_pre_ADI/results")
SAVE_CUBES = Path("/home/aosimul/noah/notebooks/results/fits_cubes")

RAW_TIF_DIR = GHOST / "phase_screens_SLM/10_10"      # raw .tif frames
RAW_BENCH_DIR = GHOST / "phase_screens_SLM/10_13"    # raw .tif frames
RUNS_BENCH = GHOST / "bench"                        # batches to characterise the bench and set up noise
LONG_RUN_FOCUS = GHOST / "long_7ro_20ws_focus"      # stacked _1-4 / _5-8, DIT = 2.814 ms with focus

DIT_LONG = 2.814e-3

#saving up to a desired amount of frames:
stop = 50

# 2. Function library

Taken from the temporal correlation notebook

In [ ]:
def wiener_khinchin_acf(stack, subtract="median"):
    """Circular temporal ACF per pixel via the Wiener-Khinchin theorem (IFFT of the PSD).

    Normalised so that lag 0 equals 1. Returns an array shaped like `stack`."""
    x = np.asarray(stack, dtype=np.float32)
    if subtract == "median":
        x = x - np.median(x, axis=0, keepdims=True)
    elif subtract == "mean":
        x = x - np.mean(x, axis=0, keepdims=True)
    P = np.abs(np.fft.rfft(x, axis=0)) ** 2
    acf = np.fft.irfft(P, n=x.shape[0], axis=0)
    zero_lag = acf[0:1]
    return np.divide(acf, zero_lag, out=np.full_like(acf, np.nan), where=zero_lag > 0)

def save_raw_and_acf(input_folder, output_folder, algo, ro = None, ws = None, key = None, count = None, stop = None):
    '''Load a np cube and save into fits file the raw data, the median-subtracted data, and the ACF.'''
    run = dict(ro=ro, ws=ws, key=key, count=count)
    stack = load_stack(input_folder, algo=algo, **run)
    stack_med_sub = stack - np.median(stack, axis = 0)
    acf = wiener_khinchin_acf(stack)
    name = "_".join(p for p in (algo, run_name(**run)) if p)   # e.g. 'pred_7ro_20ws_1key_1-4'
    fits.writeto((output_folder / f"cube_{name}_raw.fits"), stack[:stop], overwrite=True)
    fits.writeto(output_folder / f"cube_{name}_med_sub.fits", stack_med_sub[:stop], overwrite=True)
    fits.writeto(output_folder / f"cube_{name}_acf.fits", acf[:stop], overwrite=True)

# 3. Save Fits cubes

## 3.1 Phase Screens

In [ ]:
mat_paths = {
    1: "SLM_phase_screens_0.70ro_20.04ws.mat", 
    24: "now/SLM_phase_screens_0.70ro_20.04ws_24key.mat", 
    32: "now/SLM_phase_screens_0.70ro_20.04ws_32key.mat"
}
for key, mat_path in mat_paths.items():
    with h5py.File(GHOST / f"phase_screens_send/{mat_path}" , "r") as f:
        print("Original shape:", f["turb"].shape)
        stack = f["turb"][:stop, :, :]
        name = f"phase_screens_{key}key"
        fits.writeto((SAVE_CUBES / f"cube_{name}_raw.fits"), stack, overwrite=True)



## 3.2 Pre-processed data and ACF

In [ ]:
stop = 50
for ds in ["int", "pred"]:
    for key in [1,24,32]:
        save_raw_and_acf(input_folder = LONG_RUN_FOCUS, output_folder = SAVE_CUBES, algo = ds, ro=7, ws=20, key = key, count="1-4", stop = stop)

## 3.3 Bench data

In [ ]:
# Open Loop
for key in [1,24,32]:
        save_raw_and_acf(input_folder = RUNS_BENCH, output_folder = SAVE_CUBES, algo = "OL", ro=7, ws=20, key = key, count="1-4", stop = stop)

# Closed Loop
save_raw_and_acf(input_folder = RUNS_BENCH, output_folder = SAVE_CUBES, algo = "CL", count="1-4", stop = stop)


## 3.4 PCA Residuals and Fake Planet Experiment

In [ ]:
data = fits.getdata("image.fits", ext=1)


# 4. Make Fits films

In [3]:
def fits_to_video(fits_path, out_path, fps=10, cmap="inferno",
                  pmin=1, pmax=99.5, scale=1, hdu=0):
    cube = fits.getdata(fits_path, ext=hdu).astype(float)   # (n_frames, y, x)

    # one global intensity range, so brightness doesn't flicker between frames
    vmin, vmax = np.percentile(cube, [pmin, pmax])
    norm = np.clip((cube - vmin) / (vmax - vmin), 0, 1)

    lut = colormaps[cmap]
    with imageio.get_writer(out_path, fps=fps, codec="libx264",
                            quality=8, macro_block_size=1) as writer:
        for frame in norm:
            rgb = (lut(frame)[..., :3] * 255).astype(np.uint8)
            if scale > 1:                                    # integer upscaling
                rgb = rgb.repeat(scale, axis=0).repeat(scale, axis=1)
            writer.append_data(rgb)


for fits_path in sorted(
    list(SAVE_CUBES.glob("*.fits"))
):
    fits_to_video(fits_path, SAVE_CUBES / f"{fits_path.name.replace('.fits','')}.mp4", cmap = "coolwarm", fps=10, scale=4)
